# 2. Classifier evaluation
Run `python -m scripts.train_classifier` first to create `saved_models/resnet_classifier.weights.h5`.

In [ ]:
import sys
sys.path.append('..')  # repo root, so models/ and utils/ import
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay
from models.resnet_classifier import build_resnet_classifier, grad_cam
from utils.preprocessing import load_classification_data, split_dataset

In [ ]:
X, y, groups = load_classification_data('../data/download/Brain Tumor Data Set/Brain Tumor Data Set')
_, _, (X_test, y_test) = split_dataset(X, y, groups)

model = build_resnet_classifier(weights=None)
model.load_weights('../saved_models/resnet_classifier.weights.h5')

In [ ]:
probs = model.predict(X_test, batch_size=32).ravel()
pred = (probs > 0.5).astype(int)
print(classification_report(y_test, pred, target_names=['healthy', 'tumor'], digits=4))
print('ROC AUC:', roc_auc_score(y_test, probs))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=['Healthy', 'Tumor'], cmap='Blues')
plt.show()

### Grad-CAM on a test tumor scan

In [ ]:
i = np.where(y_test == 1)[0][0]
cam, p = grad_cam(model, X_test[i:i+1].astype('float32'))
plt.imshow(X_test[i]); plt.imshow(cam, cmap='inferno', alpha=0.45, extent=(0, 224, 224, 0))
plt.title(f'P(tumor) = {p:.2f}'); plt.axis('off'); plt.show()